# Projeto final: previsão de churn

## Fase 1. Análise exploratória (parte 1)

Carregamento da base, tamanho, tipos de dados e sumário estatístico.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../datasets/ecommerce_dataset.csv")

df = pd.read_csv(DATA_PATH)
df.head()

### Tamanho da base

In [ ]:
n_linhas, n_colunas = df.shape
print(f"Linhas: {n_linhas}")
print(f"Colunas: {n_colunas}")

### Tipos de dados

In [ ]:
df.info()

In [ ]:
df.dtypes

### Sumário estatístico

In [ ]:
df.describe()

In [ ]:
df.describe(include=["object", "string"])

## Fase 1. Análise exploratória (parte 2)

Nesta parte, vamos medir o desbalanceamento do alvo, observar a distribuição do tempo de relacionamento e calcular as correlações de Pearson entre as variáveis numéricas.

### Distribuição da variável-alvo

In [ ]:
distribuicao_churn = pd.DataFrame(
    {
        "clientes": df["Churn"].value_counts().sort_index(),
        "percentual": df["Churn"].value_counts(normalize=True).sort_index().mul(100),
    }
)
distribuicao_churn.index = ["Permaneceu (0)", "Saiu (1)"]
distribuicao_churn.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(
    data=df,
    x="Churn",
    hue="Churn",
    palette={0: "#4C78A8", 1: "#E45756"},
    legend=False,
    ax=ax,
)
ax.bar_label(ax.containers[0])
ax.bar_label(ax.containers[1])
ax.set(
    title="Distribuição da variável-alvo",
    xlabel="Churn (0 = permaneceu, 1 = saiu)",
    ylabel="Clientes",
)
plt.show()

### Tempo de relacionamento por classe

In [ ]:
df.groupby("Churn")["Tenure"].agg(["count", "mean", "median"]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    data=df,
    x="Tenure",
    hue="Churn",
    bins=30,
    kde=True,
    multiple="layer",
    alpha=0.45,
    palette={0: "#4C78A8", 1: "#E45756"},
    ax=ax,
)
ax.set(
    title="Distribuição do tempo de relacionamento por classe",
    xlabel="Tempo de relacionamento",
    ylabel="Clientes",
)
plt.show()

### Correlação entre variáveis numéricas

`CustomerID` fica fora do cálculo porque é apenas um identificador.

In [ ]:
variaveis_numericas = df.select_dtypes(include="number").drop(columns="CustomerID")
correlacao = variaveis_numericas.corr(method="pearson")

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    correlacao,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Correlação de Pearson entre as variáveis numéricas")
plt.tight_layout()
plt.show()

In [ ]:
(
    correlacao["Churn"]
    .drop("Churn")
    .sort_values(key=lambda valores: valores.abs(), ascending=False)
    .head(5)
)

### Leitura inicial

A base está desbalanceada: só 16,84% dos clientes saíram. A acurácia sozinha pode passar uma impressão errada.

`Tenure` é o sinal mais forte até agora, com correlação de -0,349. Quem saiu tem mediana 1; quem ficou, 10. Reclamação também puxa o churn, com correlação de 0,250, mas nenhum desses fatores sozinho fecha o caso.

Sete colunas numéricas têm valores ausentes, inclusive `Tenure`. Isso precisa ser resolvido antes da engenharia de atributos e da modelagem. O split vai manter a proporção das classes, e o balanceamento ficará só no treino.